# Notebook 06 — Jupyter and Jupytext Workflow

**Module:** 00 — Orientation  
**Notebook:** 06 of 13  
**Prerequisites:** Notebooks 01–05  
**Time estimate:** 75 minutes

This notebook explains why raw `.ipynb` files are a poor choice for Git, how Jupytext solves this, and walks through the full edit → sync → commit → re-run cycle you will use for every notebook in the program.

---
## Step 1 — Motivation

A Jupyter notebook stored as `.ipynb` is a JSON file. Every time you run a cell, the JSON changes — embedded outputs, execution counts, timestamps. When you `git diff` a notebook, you see hundreds of lines of JSON noise that have nothing to do with the code you actually changed.

This makes code review impossible and git history unreadable. It is also how large datasets accidentally get committed (saved as base64-encoded output cells).

**Jupytext + nbstripout** solve both problems:
- Jupytext pairs the `.ipynb` with a clean `.py` file (no outputs, no metadata noise) — the `.py` is what gets committed and diffed.
- nbstripout strips outputs from the `.ipynb` before it can be staged — a safety net.

**papermill** solves a different problem: reproducible re-execution. Instead of re-running a notebook manually with different parameters, `papermill` runs it programmatically and saves the output to a new file — making the run itself an artifact.

---
## Step 2 — Intuition

The mental model:

| File | Role | Committed? |
|------|------|------------|
| `notebook.py` | Source of truth — code and markdown cells, no outputs | **Yes** |
| `notebook.ipynb` | Working copy — what JupyterLab opens and runs | Optional (gitignored OR stripped) |
| `output_notebook.ipynb` | papermill output — a run with specific parameters | No (too large; re-generate on demand) |

The `.py` file is what you review, diff, and share. The `.ipynb` is what you run interactively. Jupytext keeps them synchronized.

---
## Step 3 — Biological Background

*Not applicable to this toolchain notebook.*

---
## Step 4 — Mathematical Explanation

*Not applicable to this notebook.*

---
## Step 5 — Computational Explanation

**The full workflow (one cycle per notebook):**

```
Step A: Edit
  Open notebook.ipynb in JupyterLab OR edit notebook.py in VS Code

Step B: Sync
  jupytext --sync notebook.ipynb
  → updates notebook.py to match .ipynb (if you edited .ipynb)
  → OR updates .ipynb to match .py (if you edited .py)

Step C: Run
  Open .ipynb in JupyterLab; Kernel → Restart & Run All
  Verify: does it run top-to-bottom without errors?

Step D: Commit
  git add notebooks/notebook.py       ← commit the .py, not the .ipynb
  git commit -m "module-00/nb06: ..."
  (pre-commit hook runs nbstripout automatically)

Step E: Reproduce (when needed)
  papermill notebook.ipynb output.ipynb -p SEED 42
  → re-runs the notebook with parameter SEED=42, saves output separately
```

**Why Jupytext percent format?**
The `# %%` cell marker is readable in any text editor, compatible with VS Code's
interactive Python window, and produces minimal diffs. A cell boundary change
is a one-line diff, not a 50-line JSON rearrangement.

---
## Step 6 — Python Implementation

In [ ]:
# Cell 6.1 — Verify Jupytext is installed and check its configuration
import subprocess

def run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    return r.returncode, r.stdout.strip(), r.stderr.strip()

code, out, err = run(["jupytext", "--version"])
if code == 0:
    print(f"jupytext: {out}")
else:
    print("jupytext not found. Install: pip install jupytext")

code, out, err = run(["nbstripout", "--version"])
if code == 0:
    print(f"nbstripout: {out}")
else:
    print("nbstripout not found. Install: pip install nbstripout")

code, out, err = run(["papermill", "--version"])
if code == 0:
    print(f"papermill: {out}")
else:
    print("papermill not found (optional). Install: pip install papermill")

In [ ]:
# Cell 6.2 — Demonstrate what a notebook diff looks like WITHOUT nbstripout
# (We simulate the problem; we don't actually run git diff here.)

bad_diff_example = """
  WITHOUT nbstripout — a typical .ipynb diff after running one cell:

  -   "execution_count": 3,
  +   "execution_count": 4,
       "outputs": [
  -      {
  -        "output_type": "stream",
  -        "text": ["old output here\\n"]
  -      }
  +      {
  +        "output_type": "stream",
  +        "text": ["new output here\\n"]
  +      }
       ]

  WITH nbstripout — the same change after stripping outputs:

  (no diff — outputs were stripped; only real code changes appear)
"""
print(bad_diff_example)

In [ ]:
# Cell 6.3 — Check .pre-commit-config.yaml exists and contains the right hooks
import pathlib

repo_root = pathlib.Path.cwd()
# Climb up to find the repo root (where .git lives)
while repo_root != repo_root.parent:
    if (repo_root / ".git").exists():
        break
    repo_root = repo_root.parent

config_file = repo_root / ".pre-commit-config.yaml"
if config_file.exists():
    content = config_file.read_text(encoding="utf-8")
    print(f"Found: {config_file}")
    print()
    print(content)
    has_jupytext = "jupytext" in content
    has_nbstripout = "nbstripout" in content
    print(f"\nContains jupytext hook:   {'✓' if has_jupytext else '✗ — add it'}")
    print(f"Contains nbstripout hook: {'✓' if has_nbstripout else '✗ — add it'}")
else:
    print(f"Not found: {config_file}")
    print("Create .pre-commit-config.yaml at the repo root.")
    print("See ENVIRONMENT.md for the correct content.")

In [ ]:
# Cell 6.4 — Create .pre-commit-config.yaml if missing

pre_commit_yaml = """repos:
  - repo: https://github.com/mwouts/jupytext
    rev: v1.16.0
    hooks:
      - id: jupytext
        args: [--sync]
        additional_dependencies: [jupytext]

  - repo: https://github.com/kynan/nbstripout
    rev: 0.7.1
    hooks:
      - id: nbstripout

  - repo: https://github.com/astral-sh/ruff-pre-commit
    rev: v0.4.0
    hooks:
      - id: ruff
        args: [--fix]
      - id: ruff-format
"""

if not config_file.exists():
    config_file.write_text(pre_commit_yaml, encoding="utf-8")
    print(f"Created: {config_file}")
    print("Now run: pre-commit install")
else:
    print("pre-commit config already exists — not overwriting.")
    print("To update, edit the file manually.")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — Visualize the Jupytext workflow as a diagram
workflow = """
  Jupytext Workflow Diagram
  =========================

  [VS Code: edit .py file]          [JupyterLab: run .ipynb]
         │                                    │
         └──── jupytext --sync ───────────────┘
                     │
              Keeps both files in sync
                     │
         ┌───────────┴────────────┐
         │                        │
  [.py file]               [.ipynb file]
   no outputs               has outputs
   readable diff            interactive
   git-friendly             NOT committed
         │
   git add .py
   git commit
         │
   pre-commit hooks fire:
   ├── jupytext --sync  (ensure .py matches .ipynb)
   └── nbstripout       (strip outputs from .ipynb)
         │
   Clean, readable git history ✓
"""
print(workflow)

---
## Step 8 — Exercises

*Solutions go in `exercises/06_jupytext_exercises.md`.*

**Exercise 1 — Pair a new notebook:**  
Create a new file `notebooks/test_pairing.ipynb` in JupyterLab with one markdown cell and one code cell. Run:
```bash
jupytext --set-formats ipynb,py:percent notebooks/test_pairing.ipynb
```
Open `notebooks/test_pairing.py` in VS Code. Confirm it matches the notebook. Add a new cell to the `.py` file, then run `jupytext --sync` and confirm the `.ipynb` updates.

**Exercise 2 — Verify the pre-commit hook:**  
Stage `notebooks/test_pairing.ipynb` (with outputs) and try to commit it. Confirm nbstripout runs and the outputs are stripped. Look at `git diff --staged` after the hook runs.

**Exercise 3 — Clean up:**  
Delete `test_pairing.ipynb` and `test_pairing.py`. Commit the deletion.

---
## Quiz — Active Recall

1. What is the purpose of the `# %%` marker in a Jupytext percent-format file?
2. Why is the `.py` file committed instead of the `.ipynb`?
3. What does `jupytext --sync notebook.ipynb` do when you've edited the `.ipynb`?
4. If you accidentally commit a notebook with output cells, how do you remove them from the history?
5. When would you use `papermill` instead of just re-running the notebook manually?

---
## Reflection

**Date completed:** ____________________

**Reflection:**

> *[Did the pre-commit hook work? What happened when you tried to commit a notebook with outputs? What is one thing about the Jupytext workflow that still feels unclear?]*

---
## References

- [Jupytext documentation](https://jupytext.readthedocs.io/)
- [nbstripout repository](https://github.com/kynan/nbstripout)
- [papermill documentation](https://papermill.readthedocs.io/)
- [pre-commit documentation](https://pre-commit.com/)

---
*Next notebook: `06_python_environment_and_packaging_basics.ipynb`*